<a href="https://colab.research.google.com/github/Lornatt88/maai902-nlp-solution/blob/main/notebooks/06_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 6: Demo
The Fine Print Detector as a working app: paste a Terms of Service, and it flags the unfair clauses, names their type, and explains them in plain English.

## 1. Setup
Loads the trained Legal-BERT model (run A) and the knowledge base from Google Drive.

In [2]:
!pip install -q transformers accelerate bitsandbytes sentence-transformers gradio nltk

import torch, numpy as np, pandas as pd
from google.colab import drive
from transformers import AutoTokenizer, AutoModelForSequenceClassification

drive.mount("/content/drive")
save_dir = "/content/drive/MyDrive/fine_print_detector"
print("GPU:", torch.cuda.is_available())

kb = pd.read_csv(f"{save_dir}/knowledge_base.csv")          # the 87 explanations

clf_dir = f"{save_dir}/models/legal_bert_A"                 # trained model
clf_tok = AutoTokenizer.from_pretrained(clf_dir)
clf = AutoModelForSequenceClassification.from_pretrained(clf_dir).to("cuda").eval()
names = [clf.config.id2label[i] for i in range(8)]

print("Explanations:", len(kb))
print("Types:", names)

Mounted at /content/drive
GPU: True


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Explanations: 87
Types: ['Limitation of liability', 'Unilateral termination', 'Unilateral change', 'Content removal', 'Contract by using', 'Choice of law', 'Jurisdiction', 'Arbitration']


## 2. Loading the LLM
Loads Qwen2.5-7B-Instruct, compressed to 4-bit so it fits on Colab's GPU. It is used for the extractive summaries of long sentences and for the explanations.


In [3]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

llm_name = "Qwen/Qwen2.5-7B-Instruct"
llm_tok = AutoTokenizer.from_pretrained(llm_name)
llm = AutoModelForCausalLM.from_pretrained(
    llm_name,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16),
    device_map="auto",
)

def ask_llm(prompt, max_new_tokens=250):
    """Send one message to the LLM and return its answer (no randomness, so answers are repeatable)."""
    messages = [{"role": "user", "content": prompt}]
    text = llm_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = llm_tok(text, return_tensors="pt").to(llm.device)
    out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return llm_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print(ask_llm("Say hello in one short sentence."))

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Hello!


## 3. Retrieval helpers
Loads MiniLM and the reranker, and defines retrieve(): it finds the 3 expert explanations closest to a flagged clause, within its unfair type (the final setup from Notebook 5).

In [4]:
from sentence_transformers import SentenceTransformer, CrossEncoder

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
kb_vecs = embedder.encode(kb["text"].tolist(), normalize_embeddings=True)

def retrieve(clause, clause_type, k=3, shortlist=30):
    """Shortlist explanations of this type with MiniLM, re-order them with the reranker, return the top k."""
    mask = (kb["type"] == clause_type).values
    sub = kb[mask]
    q = embedder.encode([clause], normalize_embeddings=True)[0]
    first = np.argsort(-(kb_vecs[mask] @ q))[:shortlist]
    rs = reranker.predict([(clause, sub["text"].iloc[c]) for c in first])
    order = np.argsort(-rs)[:k]
    return sub.iloc[first[order]]

print(retrieve("we may terminate your account at any time without notice .", "Unilateral termination")[["name", "severity"]])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

          name            severity
64     dormant  potentially unfair
80  any_reason      clearly unfair
68    payments  potentially unfair


## 4. Text preparation and extractive summaries
Splits pasted text into sentences, removes fragments under 5 words (Lippi et al., 2019), and formats each sentence like the training data. Sentences longer than 128 tokens are shortened with the same extractive summary method as Notebook 2b (the LLM picks exact phrases, and the least important are dropped until it fits). The user still sees the original sentence.

In [5]:
import re, nltk
nltk.download("punkt", quiet=True); nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import sent_tokenize

# ---------- Preparing pasted text ----------
def to_dataset_style(s):
    """Format like the training data: lowercase, the dataset's quote marks, spaces around punctuation."""
    s = s.lower().strip().replace("“", '"').replace("”", '"').replace("’", "'")
    s = re.sub(r'"([^"]*)"', r" `` \1 '' ", s)
    s = re.sub(r"(\w)'s\b", r"\1 's", s)
    s = re.sub(r"\s*([,;:()])\s*", r" \1 ", s)
    s = re.sub(r"\s*\.$", " .", s)
    return re.sub(r"\s+", " ", s).strip()

def prepare(text):
    """Split into sentences and keep those with at least 5 words."""
    sentences = []
    for line in text.split("\n"):
        sentences += sent_tokenize(line)
    return [s.strip() for s in sentences if len(s.split()) >= 5]

# ---------- Extractive summary (same method as Notebook 2b) ----------
PROMPT = """Make an extractive summary of this clause from an online Terms of Service.

List the most important phrases from the clause, one per line.
Rules:
- Copy each phrase exactly as it appears in the clause. Never change, reword or add words.
- Choose the phrases that say what the company or the user can or cannot do: rights, obligations, limits on liability, changes to the terms, account termination, content removal, how disputes are handled, and which laws or courts apply.
- Skip examples, long lists and background.
- All phrases together must be at most 70 words.
- Return only the phrases, one per line, without numbers or quotation marks.

Example:
Clause: we may , at our sole discretion and for any reason , including but not limited to violations of these terms or suspected fraud , suspend or terminate your account at any time without notice .
Phrases:
we may , at our sole discretion and for any reason
suspend or terminate your account at any time without notice

Clause: {clause}
Phrases:"""

MAIN_PROMPT = """These phrases come from a clause in an online Terms of Service.
Which ONE phrase states the main rule of the clause (who does or promises what)?
Copy that phrase exactly and return only it.

Phrases:
{phrases}"""

DROP_PROMPT = """These phrases come from a clause in an online Terms of Service.
Which ONE phrase is the least important for understanding what the company or the user can or cannot do?
Copy that phrase exactly and return only it.

Phrases:
{phrases}"""

def words(s):
    return re.findall(r"[a-z0-9]+", s.lower())

def clean_line(line):
    l = line.strip(" -*•")
    if len(l) > 1 and l[0] in '"“' and l[-1] in '"”':
        l = l[1:-1]
    return l

def join(found):
    return " ... ".join(p for _, p in found)

def exact_pieces(phrase, original):
    """Split a phrase into pieces that each appear exactly in the original, in order; None if a word was added or changed."""
    ws = phrase.split()
    padded = " " + original + " "
    pieces, start, pos = [], 0, 0
    while start < len(ws):
        for k in range(len(ws), start, -1):
            cand = " " + " ".join(ws[start:k]) + " "
            p = padded.find(cand, pos)
            if p != -1:
                pieces.append((p, " ".join(ws[start:k])))
                pos = p + len(cand) - 1
                start = k
                break
        else:
            return None
    return pieces

def get_phrases(clause, original):
    answer = ask_llm(PROMPT.format(clause=clause), max_new_tokens=250)
    found = []
    for line in [l for l in answer.splitlines() if l.strip()]:
        phrase = to_dataset_style(clean_line(line))
        pieces = exact_pieces(phrase, original)
        if pieces and len(words(phrase)) >= 2:
            for item in pieces:
                if item[1] not in [p for _, p in found]:
                    found.append(item)
    found.sort()
    return found

def pick(prompt, options):
    listing = "\n".join(p for _, p in options)
    answer = to_dataset_style(clean_line(ask_llm(prompt.format(phrases=listing), max_new_tokens=80)))
    for item in options:
        if item[1] == answer:
            return item
    return None

def too_long(found):
    return len(clf_tok(join(found))["input_ids"]) > 128

def shorten(text):
    """Exact phrases, lock the main rule, drop the least important until it fits in 128 tokens. None if it fails."""
    found = get_phrases(text, text)
    if len(found) == 1 and too_long(found):
        sub = get_phrases(found[0][1], text)
        if len(sub) > 1:
            found = sub
    if not found:
        return None
    if too_long(found):
        main = pick(MAIN_PROMPT, found) or found[0]
        while too_long(found):
            others = [x for x in found if x != main]
            if not others:
                return None
            drop = pick(DROP_PROMPT, others) or max(others, key=lambda x: len(x[1]))
            found = [x for x in found if x != drop]
    return join(found) if len(words(join(found))) >= 5 else None

def model_input(sentence):
    """What Legal-BERT reads: the formatted sentence, or its extractive summary if it is over 128 tokens."""
    t = to_dataset_style(sentence)
    if len(clf_tok(t)["input_ids"]) > 128:
        t = shorten(t) or t          # backup: standard cut at 128 if the summary fails
    return t

# Test on one long clause
long_clause = ("To the fullest extent permitted by applicable law, in no event will the company, its affiliates, "
               "employees, licensors or service providers be liable for any indirect, consequential, exemplary, "
               "incidental, special or punitive damages, including, without limitation, loss of profits, whether "
               "incurred directly or indirectly, or any loss of data, use, goodwill, or other intangible losses, "
               "resulting from: (i) your access to or use of or inability to access or use the services, (ii) the "
               "conduct or content of other users or third parties on, through, or following use of the services; "
               "or (iii) unauthorized access, use or alteration of your content, even if the company has been "
               "advised of the possibility of such damages.")
print("Tokens before:", len(clf_tok(to_dataset_style(long_clause))["input_ids"]))
short = model_input(long_clause)
print("Tokens after:", len(clf_tok(short)["input_ids"]))
print("SUMMARY:", short)

Tokens before: 153
Tokens after: 110
SUMMARY: in no event will the company , its affiliates , employees , licensors or service providers be liable for any indirect , consequential , exemplary , incidental , special or punitive damages ... including , without limitation , loss of profits , whether incurred directly or indirectly ... or any loss of data , use , goodwill , or other intangible losses ... resulting from : ( i ) your access to or use of or inability to access or use the services ... ( iii ) unauthorized access , use or alteration of your content


## 5. Detection
Runs Legal-BERT on every prepared sentence (or its extractive summary if it is long). A sentence is flagged for every unfair type with a score above 0.5, and the types are sorted from most to least confident.

In [6]:
def detect(sentences, threshold=0.5):
    """Run Legal-BERT on each sentence and return the flagged ones with their types and confidence."""
    texts = [model_input(s) for s in sentences]
    enc = clf_tok(texts, truncation=True, max_length=128, padding=True, return_tensors="pt").to("cuda")
    with torch.no_grad():
        probs = torch.sigmoid(clf(**enc).logits).cpu().numpy()
    flagged = []
    for s, p in zip(sentences, probs):
        types = [(names[i], float(p[i])) for i in np.argsort(-p) if p[i] > threshold]
        if types:
            flagged.append({"sentence": s, "types": types})
    return flagged

sample = """Terms of Use
We may terminate your account at any time without notice.
You can contact our support team by email.
By using the app, you agree to these terms.
Any dispute will be resolved by binding arbitration."""

for f in detect(prepare(sample) + [long_clause]):
    print(f["sentence"][:80], "->", [(t, round(p, 2)) for t, p in f["types"]])

We may terminate your account at any time without notice. -> [('Unilateral termination', 0.99)]
By using the app, you agree to these terms. -> [('Contract by using', 0.99)]
Any dispute will be resolved by binding arbitration. -> [('Arbitration', 0.94)]
To the fullest extent permitted by applicable law, in no event will the company, -> [('Limitation of liability', 0.99)]


## 6. Explanations
For each flagged clause, retrieves the 3 closest expert explanations and lets the LLM write a plain-English explanation using only those (the final prompt from Notebook 5). If the LLM replies "NO MATCH", a short general explanation for the type is shown instead.

In [7]:
EXPLAIN_PROMPT = """You help everyday users understand unfair clauses in online Terms of Service.

Clause: {clause}
Type: {clause_type}

Expert explanations:
{sources}

Rules:
- Use ONLY the expert explanations above. Do not add other legal facts.
- If none of the expert explanations matches what this clause says, reply only: NO MATCH
- Users cannot negotiate these terms, so only suggest realistic actions, such as: keeping screenshots or records, backing up their content, comparing other services, or contacting a consumer protection organization. Only mention an opt-out if the clause itself mentions one.
- Do not give legal advice.

Answer in exactly this format:
Why this may be unfair: <2 short sentences in plain English>
What you can do: <1 or 2 short tips>"""

BACKUP = {
    "Arbitration": "the clause requires disputes to be settled by an arbitrator instead of a court.",
    "Unilateral change": "the clause lets the provider change the terms or the service on its own.",
    "Content removal": "the clause lets the provider remove the user's content on its own.",
    "Limitation of liability": "the clause limits or excludes the provider's responsibility for harm or losses.",
    "Unilateral termination": "the clause lets the provider suspend or close the user's account on its own.",
    "Jurisdiction": "the clause decides which courts handle disputes, which may be far from where the user lives.",
    "Choice of law": "the clause decides which country's law applies, which may not be the user's own.",
    "Contract by using": "the clause says that simply using the service means accepting the terms.",
}

def explain(clause, clause_type):
    """Retrieve 3 expert explanations, let the LLM explain from them, or use the backup if none fits."""
    found = retrieve(model_input(clause), clause_type)
    sources = "\n".join(f"- {t}" for t in found["text"])
    answer = ask_llm(EXPLAIN_PROMPT.format(clause=clause, clause_type=clause_type, sources=sources))
    used_backup = "NO MATCH" in answer.upper()
    if used_backup:
        b = BACKUP[clause_type]
        answer = f"Why this may be unfair: {b[0].upper()}{b[1:]}\n(General explanation for this type.)"
    return answer, found, used_backup

for f in detect(prepare(sample)):
    t = f["types"][0][0]
    answer, found, used_backup = explain(f["sentence"], t)
    print(f"===== {t} {'(BACKUP)' if used_backup else ''} =====")
    print(f["sentence"])
    print(answer)
    print("Sources:", ", ".join(found["name"]), "\n")

===== Unilateral termination  =====
We may terminate your account at any time without notice.
Why this may be unfair: This clause allows the service provider to end your account with no notice and for any reason, which can be arbitrary and without specific guidelines.

What you can do: Keep screenshots or records of your interactions with the service provider, and consider comparing similar services that might offer more transparent policies regarding account termination.
Sources: dormant, any_reason, payments 

===== Contract by using  =====
By using the app, you agree to these terms.
Why this may be unfair: This clause binds you to the terms without requiring you to explicitly accept them, which means you might agree to conditions even if you haven't read them thoroughly.

What you can do: Keep screenshots or records of your agreement to the terms and compare similar services to find more favorable options.
Sources: use_is_consent 

===== Arbitration  =====
Any dispute will be resolv

## 7. Test on a real Terms of Service
Runs the full pipeline on part of a real, current Terms of Service (not in the dataset): split, detect, explain. It shows how many sentences were checked, how many were flagged, and how long it took. The flagged clauses are then checked by hand.

In [9]:
import time

real_tos = """Termination. You are free to stop using our Services at any time. We reserve the right to suspend or terminate your access to our Services or delete your account if we determine:

You breached these Terms or our Usage Policies⁠⁠.

We must do so to comply with the law.

Your use of our Services could cause risk or harm to OpenAI, our users, or anyone else.

We also may terminate your account if it has been inactive for over a year and you do not have a paid account. If we do, we will provide you with advance notice.

Appeals. If you believe we have suspended or terminated your account in error, you can file an appeal with us by contacting our Support team⁠⁠(opens in a new window). OUR SERVICES ARE PROVIDED “AS IS.” EXCEPT TO THE EXTENT PROHIBITED BY LAW, WE AND OUR AFFILIATES AND LICENSORS MAKE NO WARRANTIES (EXPRESS, IMPLIED, STATUTORY OR OTHERWISE) WITH RESPECT TO THE SERVICES, AND DISCLAIM ALL WARRANTIES INCLUDING, BUT NOT LIMITED TO, WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE, SATISFACTORY QUALITY, NON-INFRINGEMENT, AND QUIET ENJOYMENT, AND ANY WARRANTIES ARISING OUT OF ANY COURSE OF DEALING OR TRADE USAGE. WE DO NOT WARRANT THAT THE SERVICES WILL BE UNINTERRUPTED, ACCURATE OR ERROR FREE, OR THAT ANY CONTENT WILL BE SECURE OR NOT LOST OR ALTERED.

YOU ACCEPT AND AGREE THAT ANY USE OF OUTPUTS FROM OUR SERVICE IS AT YOUR SOLE RISK AND YOU WILL NOT RELY ON OUTPUT AS A SOLE SOURCE OF TRUTH OR FACTUAL INFORMATION, OR AS A SUBSTITUTE FOR PROFESSIONAL ADVICE. NEITHER WE NOR ANY OF OUR AFFILIATES OR LICENSORS WILL BE LIABLE FOR ANY INDIRECT, INCIDENTAL, SPECIAL, CONSEQUENTIAL, OR EXEMPLARY DAMAGES, INCLUDING DAMAGES FOR LOSS OF PROFITS, GOODWILL, USE, OR DATA OR OTHER LOSSES, EVEN IF WE HAVE BEEN ADVISED OF THE POSSIBILITY OF SUCH DAMAGES. OUR AGGREGATE LIABILITY UNDER THESE TERMS WILL NOT EXCEED ​​THE GREATER OF THE AMOUNT YOU PAID FOR THE SERVICE THAT GAVE RISE TO THE CLAIM DURING THE 12 MONTHS BEFORE THE LIABILITY AROSE OR ONE HUNDRED DOLLARS ($100). THE LIMITATIONS IN THIS SECTION APPLY ONLY TO THE MAXIMUM EXTENT PERMITTED BY APPLICABLE LAW.

Some countries and states do not allow the disclaimer of certain warranties or the limitation of certain damages, so some or all of the terms above may not apply to you, and you may have additional rights. In that case, these Terms only limit our responsibilities to the maximum extent permissible in your country of residence.

OPENAI’S AFFILIATES, SUPPLIERS, LICENSORS, AND DISTRIBUTORS ARE INTENDED THIRD PARTY BENEFICIARIES OF THIS SECTION."""

start = time.time()
sentences = prepare(real_tos)
flagged = detect(sentences)
print(f"Sentences checked: {len(sentences)} | Flagged: {len(flagged)}\n")

for f in flagged:
    t, conf = f["types"][0]
    answer, found, used_backup = explain(f["sentence"], t)
    print(f"===== {t} ({conf:.0%} sure) {'(BACKUP)' if used_backup else ''} =====")
    print(f["sentence"])
    print(answer)
    print("Sources:", ", ".join(found["name"]), "\n")

print(f"Time: {time.time() - start:.0f} seconds")

Sentences checked: 17 | Flagged: 5

===== Unilateral termination (100% sure)  =====
We reserve the right to suspend or terminate your access to our Services or delete your account if we determine:
Why this may be unfair: This clause gives the service provider complete control over when and why they can suspend or terminate your account without any clear guidelines or due process for the user.
What you can do: Keep screenshots or records of any communications with the service provider regarding your account, and consider comparing similar services to find one with more transparent policies.
Sources: shutdown, gen_rights_violation, protect_rights 

===== Unilateral termination (99% sure)  =====
We also may terminate your account if it has been inactive for over a year and you do not have a paid account.
Why this may be unfair: This clause allows the company to terminate your account simply because it hasn't been used for a year, even if you haven't missed any payments. It doesn't conside

## 8. The app
A web interface built with Gradio. Users paste a Terms of Service and get the flagged clauses as colored cards (type, confidence, plain-English explanation, expert sources), plus the full text with flagged clauses highlighted. share=True creates a public link that opens in any browser while this notebook runs.

In [11]:
import gradio as gr, html
from collections import Counter

COLORS = {
    "Limitation of liability": "#b4533b", "Unilateral termination": "#c2703d",
    "Unilateral change": "#b8892f",       "Content removal": "#7d5a8c",
    "Contract by using": "#4f6d8f",       "Choice of law": "#4f8a83",
    "Jurisdiction": "#6b8a4f",            "Arbitration": "#a8546f",
}
PLAIN = {
    "Limitation of liability": "“It’s not our fault if something goes wrong.”",
    "Unilateral termination": "“We can close your account whenever we decide.”",
    "Unilateral change": "“We can change these rules whenever we want.”",
    "Content removal": "“We can delete your posts or files whenever we want.”",
    "Contract by using": "“Just using the app means you agreed to everything.”",
    "Choice of law": "“Our country’s laws apply, not yours.”",
    "Jurisdiction": "“To sue us, you must go to our courts, far from you.”",
    "Arbitration": "“Disputes go to a private arbitrator, not a real court.”",
}

def badge(t, conf=None):
    label = f"{t} · {conf:.0%}" if conf is not None else t
    return f'<span class="badge" style="--c:{COLORS[t]}">{label}</span>'

def format_answer(answer):
    out = []
    for line in html.escape(answer).split("\n"):
        line = line.strip()
        if not line:
            continue
        for label in ["Why this may be unfair:", "What you can do:"]:
            if line.startswith(label):
                line = f"<span class='lbl'>{label}</span>{line[len(label):]}"
        out.append(f"<p>{line}</p>")
    return "".join(out)

def analyse(text, progress=gr.Progress()):
    if not text or len(text.split()) < 5:
        msg = "<p class='empty'>Paste some Terms of Service text to begin.</p>"
        return msg, msg

    progress(0, desc="Reading the fine print...")
    sentences = prepare(text)
    flagged = detect(sentences)
    by_sentence = {f["sentence"]: f for f in flagged}

    counts = Counter(f["types"][0][0] for f in flagged)
    chips = " ".join(badge(t) + f"<span class='count'>×{n}</span>" for t, n in counts.most_common())
    out = [f"""<div class='summary fade'>
        <div class='big'>{len(flagged)}</div>
        <div class='sumtext'>potentially unfair clause{'s' if len(flagged) != 1 else ''}<br>
        <span>found in {len(sentences)} sentences</span></div>
        <div class='chips'>{chips}</div></div>"""]
    if not flagged:
        out.append("<p class='empty'>No potentially unfair clauses were detected. ✨</p>")

    for i, f in enumerate(flagged):
        progress((i + 1) / len(flagged), desc=f"Explaining clause {i + 1} of {len(flagged)}...")
        t, conf = f["types"][0]
        answer, found, used_backup = explain(f["sentence"], t)
        badges = " ".join(badge(tt, cc) for tt, cc in f["types"])
        sources = "".join(
            f"<li><b>{html.escape(r['name'])}</b> · <i>{html.escape(str(r['severity']))}</i> · "
            f"<span class='src'>{html.escape(r['source'])}</span><br>{html.escape(r['text'])}</li>"
            for _, r in found.iterrows())
        note = ("<p class='note'>No closely matching expert explanation was found, "
                "so a general explanation is shown.</p>") if used_backup else ""
        out.append(f"""<div class='card' style='--c:{COLORS[t]}; animation-delay:{0.12 * (i + 1):.2f}s'>
            <div class='badges'>{badges}</div>
            <blockquote>{html.escape(f['sentence'])}</blockquote>
            <div class='answer'>{format_answer(answer)}{note}</div>
            <details><summary>Expert sources</summary><ul>{sources}</ul></details></div>""")

    doc = " ".join(
        f"<mark style='--c:{COLORS[by_sentence[s]['types'][0][0]]}' "
        f"title='{by_sentence[s]['types'][0][0]}'>{html.escape(s)}</mark>"
        if s in by_sentence else html.escape(s) for s in sentences)
    legend = " ".join(badge(t) for t in COLORS)
    return "".join(out), f"<div class='legend fade'>{legend}</div><div class='doc fade'>{doc}</div>"

ABOUT = "<div class='about fade'>" + """
<h2>How it works</h2>
<div class='steps'>
  <div class='step'><div class='num'>1</div><b>Read</b><p>The text is split into sentences. Very long ones get an extractive summary using only their original words.</p></div>
  <div class='step'><div class='num'>2</div><b>Detect</b><p>Legal-BERT, fine-tuned on expert-labelled Terms of Service, checks each sentence for 8 types of unfair clauses.</p></div>
  <div class='step'><div class='num'>3</div><b>Explain</b><p>The closest expert legal explanations are retrieved, and an LLM rewrites them in plain English.</p></div>
  <div class='step'><div class='num'>4</div><b>Review</b><p>You see the original clause, the explanation and the expert sources side by side.</p></div>
</div>
<h2>The 8 types, in plain words</h2>
<div class='types'>""" + "".join(
    f"<div class='type'>{badge(t)}<p>{PLAIN[t]}</p></div>" for t in COLORS) + """</div>
<h2>Key results</h2>
<div class='stats'>
  <div class='stat'><div class='val'>82.8</div><div>macro F1, Legal-BERT<br><span>(test set, 3 runs)</span></div></div>
  <div class='stat'><div class='val'>76.5</div><div>macro F1, word-based<br><span>baseline (TF-IDF + SVM)</span></div></div>
  <div class='stat'><div class='val'>67.8%</div><div>right expert explanation<br><span>in the top 3 (random: 31%)</span></div></div>
  <div class='stat'><div class='val'>9,414</div><div>expert-labelled sentences<br><span>from 50 Terms of Service</span></div></div>
</div>
<p class='small'>Data: UNFAIR-ToS (Lippi et al., 2019; Chalkidis et al., 2022). Expert explanations: Ruggeri et al. (2022).</p>
</div>"""

CSS = """
@import url('https://fonts.googleapis.com/css2?family=Cormorant+Garamond:wght@500;600;700&family=Lora:ital,wght@0,400;0,600;1,400&display=swap');

:root {--ink:#2f2924; --muted:#7a6e63; --paper:#fffcf7; --sand:#efe6da; --line:#e3d7c7; --accent:#8c5a3c;}
.gradio-container {background: radial-gradient(circle at 15% 0%, #fbf5ec 0%, #f4ece1 45%, #efe5d8 100%) !important;
                   font-family:'Lora', Georgia, serif !important; color:var(--ink) !important;}
footer {visibility:hidden;}

@keyframes fadeUp {from {opacity:0; transform:translateY(14px);} to {opacity:1; transform:none;}}
@keyframes glow {0%,100% {box-shadow:0 0 0 0 transparent;} 50% {box-shadow:0 0 0 3px color-mix(in srgb, var(--c) 18%, transparent);}}
.fade {animation:fadeUp .7s ease both;}

.header {text-align:center; padding:34px 12px 6px; animation:fadeUp .9s ease both;}
.header .eyebrow {letter-spacing:.3em; text-transform:uppercase; font-size:.72rem; color:var(--accent);}
.header h1 {font-family:'Cormorant Garamond', serif; font-size:3.6rem; font-weight:700; margin:6px 0 0; color:var(--ink);}
.header p {color:var(--muted); margin:8px auto 0; max-width:620px; font-size:1.05rem; font-style:italic;}
.header .rule {width:70px; height:2px; background:var(--accent); margin:18px auto 0; border-radius:2px;}

.summary {display:flex; align-items:center; gap:18px; flex-wrap:wrap; padding:20px 24px; border-radius:18px;
          background:var(--paper); border:1px solid var(--line); margin-bottom:18px;
          box-shadow:0 8px 24px -16px rgba(80,55,30,.35);}
.summary .big {font-family:'Cormorant Garamond', serif; font-size:3.4rem; font-weight:700; line-height:1; color:var(--accent);}
.summary .sumtext {font-size:1.05rem;} .summary .sumtext span {color:var(--muted); font-size:.9rem;}
.chips {flex-basis:100%;}
.count {margin:0 12px 0 4px; color:var(--muted); font-size:.85rem;}

.badge {display:inline-block; color:var(--c); background:color-mix(in srgb, var(--c) 12%, white);
        border:1px solid color-mix(in srgb, var(--c) 35%, white); padding:3px 11px; border-radius:999px;
        font-size:.78rem; font-weight:600; margin:2px 3px 2px 0; font-family:'Lora', serif;}

.card {border-left:5px solid var(--c); border-radius:16px; padding:18px 22px; margin-bottom:16px;
       background:var(--paper); border-top:1px solid var(--line); border-right:1px solid var(--line);
       border-bottom:1px solid var(--line); box-shadow:0 10px 26px -18px rgba(80,55,30,.4);
       animation:fadeUp .7s ease both; transition:transform .25s ease, box-shadow .25s ease;}
.card:hover {transform:translateY(-3px); box-shadow:0 16px 32px -18px rgba(80,55,30,.5);}
.card blockquote {margin:12px 0; padding:12px 16px; border-radius:10px; font-style:italic; color:#4a4037;
                  background:var(--sand); border:none; line-height:1.6;}
.answer p {margin:8px 0; line-height:1.65;}
.lbl {font-family:'Cormorant Garamond', serif; font-weight:700; font-size:1.15rem; color:var(--c); margin-right:4px;}
.card details {margin-top:10px; padding-top:10px; border-top:1px dashed var(--line);}
.card summary {cursor:pointer; color:var(--accent); font-weight:600;}
.card li {margin:8px 0; font-size:.9rem; color:#4a4037; line-height:1.5;}
.src {color:var(--muted);}
.note {font-size:.85rem; color:var(--muted); font-style:italic;}

.doc {line-height:2; padding:18px 22px; background:var(--paper); border-radius:16px; border:1px solid var(--line);}
.doc mark {color:inherit; background:color-mix(in srgb, var(--c) 16%, transparent);
           border-bottom:2px solid var(--c); border-radius:4px; padding:1px 3px; animation:glow 2.6s ease-in-out 2;}
.legend {margin-bottom:12px;}
.empty {color:var(--muted); padding:28px; text-align:center; font-style:italic;}

.about h2 {font-family:'Cormorant Garamond', serif; font-size:1.9rem; margin:18px 0 10px; color:var(--ink);}
.steps, .stats {display:grid; grid-template-columns:repeat(auto-fit, minmax(170px, 1fr)); gap:14px;}
.step, .stat, .type {background:var(--paper); border:1px solid var(--line); border-radius:16px; padding:16px;
                     transition:transform .25s ease;}
.step:hover, .stat:hover, .type:hover {transform:translateY(-3px);}
.step .num {width:34px; height:34px; border-radius:50%; background:var(--accent); color:white; display:flex;
            align-items:center; justify-content:center; font-family:'Cormorant Garamond', serif; font-weight:700;
            font-size:1.2rem; margin-bottom:8px;}
.step p, .type p {color:#5a4f45; font-size:.9rem; margin:6px 0 0; line-height:1.5;}
.types {display:grid; grid-template-columns:repeat(auto-fit, minmax(230px, 1fr)); gap:12px;}
.stat .val {font-family:'Cormorant Garamond', serif; font-size:2.4rem; font-weight:700; color:var(--accent);}
.stat span {color:var(--muted); font-size:.85rem;}
.small {color:var(--muted); font-size:.85rem; margin-top:16px; font-style:italic;}

.footer {text-align:center; color:var(--muted); font-size:.85rem; padding:18px; font-style:italic;}
button.primary {transition:transform .2s ease, box-shadow .2s ease !important;}
button.primary:hover {transform:translateY(-2px); box-shadow:0 10px 20px -10px rgba(47,41,36,.6) !important;}
"""

FORCE_LIGHT = """() => {
  const url = new URL(window.location);
  if (url.searchParams.get('__theme') !== 'light') {
    url.searchParams.set('__theme', 'light'); window.location.href = url.href;
  }
}"""

EXAMPLE = """We may terminate your account at any time without notice.
You can contact our support team by email.
By using the app, you agree to these terms.
We may change these terms from time to time, and the updated version will be posted on this page.
Any dispute will be resolved by binding arbitration in San Francisco, California."""

theme = gr.themes.Base(
    primary_hue="orange", neutral_hue="stone",
    font=[gr.themes.GoogleFont("Lora"), "Georgia", "serif"],
).set(
    body_background_fill="#f4ece1", block_background_fill="#fffcf7", block_border_color="#e3d7c7",
    input_background_fill="#fffcf7", body_text_color="#2f2924", block_label_text_color="#7a6e63",
    button_primary_background_fill="#2f2924", button_primary_background_fill_hover="#8c5a3c",
    button_primary_text_color="#fffcf7", block_radius="16px",
)

with gr.Blocks(theme=theme, css=CSS, js=FORCE_LIGHT, title="Fine Print Detector") as app:
    gr.HTML("""<div class='header'><div class='eyebrow'>Terms of Service, read for you</div>
        <h1>The Fine Print Detector</h1>
        <p>Paste any Terms of Service. We highlight the clauses that may be unfair and explain them in plain English.</p>
        <div class='rule'></div></div>""")
    with gr.Tabs():
        with gr.Tab("Check a document"):
            with gr.Row():
                with gr.Column(scale=5):
                    inp = gr.Textbox(lines=16, label="Terms of Service", placeholder="Paste the text here...")
                    btn = gr.Button("Check the fine print", variant="primary", size="lg")
                    gr.Examples([[EXAMPLE]], inputs=inp, label="Try an example")
                with gr.Column(scale=7):
                    with gr.Tabs():
                        with gr.Tab("Flagged clauses"):
                            out_cards = gr.HTML("<p class='empty'>Your results will appear here.</p>")
                        with gr.Tab("Full text"):
                            out_doc = gr.HTML("<p class='empty'>The highlighted text will appear here.</p>")
        with gr.Tab("About"):
            gr.HTML(ABOUT)
    gr.HTML("""<div class='footer'>Clauses are flagged for review, not as legal judgments. This tool does not give legal advice.<br>
        Detection: Legal-BERT fine-tuned on UNFAIR-ToS · Explanations grounded in expert rationales (Ruggeri et al., 2022)</div>""")
    btn.click(analyse, inp, [out_cards, out_doc])

app.launch(share=True)

/tmp/ipykernel_1949/818720166.py:195: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css, js. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=theme, css=CSS, js=FORCE_LIGHT, title="Fine Print Detector") as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://eb0d9457b327d46113.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
